# 🚗 CO2 Emissions Canada — ML Regression Project by Alok

**Dataset:** CO2 Emissions by Vehicle (Canada)  
**Goal:** Predict CO2 emissions (g/km) based on vehicle features like engine size, cylinders, and fuel consumption  
**Algorithms used:** Linear Regression, Ridge Regression, Decision Tree Regressor, Random Forest Regressor

## Step 1 — Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


## Step 2 — Load the Dataset

In [ ]:
df = pd.read_csv('CO2 Emissions_Canada.csv')
print("Shape:", df.shape)
df.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'CO2_Emissions_Canada.csv'

## Step 3 — Data Exploration

In [ ]:
print("Column names:")
print(df.columns.tolist())
print()
print("Data types:")
print(df.dtypes)

In [ ]:
print("Basic statistics:")
df.describe()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

In [ ]:
# Check unique values in categorical columns
print("Unique Makes:", df['Make'].nunique())
print("Unique Models:", df['Model'].nunique())
print("Unique Vehicle Classes:", df['Vehicle Class'].nunique())
print("Unique Fuel Types:", df['Fuel Type'].unique())
print("Unique Transmissions:", df['Transmission'].nunique())

## Step 4 — Data Cleaning

In [ ]:
# Strip whitespace from column names and string values
df.columns = df.columns.str.strip()
df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Drop duplicate rows if any
before = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Removed {before - df.shape[0]} duplicate rows")

# Check target column
print("\nCO2 Emissions range:", df['CO2 Emissions(g/km)'].min(), "to", df['CO2 Emissions(g/km)'].max())
print("Any nulls in target?", df['CO2 Emissions(g/km)'].isnull().sum())

In [ ]:
# Encode categorical columns using LabelEncoder
cat_cols = ['Make', 'Model', 'Vehicle Class', 'Transmission', 'Fuel Type']
le = LabelEncoder()

for col in cat_cols:
    df[col + '_encoded'] = le.fit_transform(df[col])
    
print("Encoding done! New columns added:")
print([c for c in df.columns if '_encoded' in c])

## Step 5 — Data Visualisation

In [ ]:
# CO2 Emissions distribution
plt.figure(figsize=(8, 4))
plt.hist(df['CO2 Emissions(g/km)'], bins=40, color='#C8906A', edgecolor='black', alpha=0.8)
plt.title('Distribution of CO2 Emissions (g/km)', fontsize=14)
plt.xlabel('CO2 Emissions (g/km)')
plt.ylabel('Number of Vehicles')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 most common makes
plt.figure(figsize=(10, 5))
top_makes = df['Make'].value_counts().head(10)
top_makes.plot(kind='bar', color='#4A6C8C', edgecolor='black')
plt.title('Top 10 Car Makes in Dataset', fontsize=14)
plt.xlabel('Make')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# CO2 emissions by fuel type
plt.figure(figsize=(9, 5))
df.boxplot(column='CO2 Emissions(g/km)', by='Fuel Type',
           boxprops=dict(color='#4A8C6A'),
           medianprops=dict(color='red', linewidth=2))
plt.title('CO2 Emissions by Fuel Type', fontsize=14)
plt.suptitle('')
plt.xlabel('Fuel Type')
plt.ylabel('CO2 Emissions (g/km)')
plt.tight_layout()
plt.show()

In [ ]:
# Engine size vs CO2 emissions scatter
plt.figure(figsize=(8, 5))
plt.scatter(df['Engine Size(L)'], df['CO2 Emissions(g/km)'],
            alpha=0.3, color='#8C6A9E', s=20)
plt.title('Engine Size vs CO2 Emissions', fontsize=14)
plt.xlabel('Engine Size (L)')
plt.ylabel('CO2 Emissions (g/km)')
plt.tight_layout()
plt.show()

In [ ]:
# Fuel consumption (combined) vs CO2 — should be very correlated
plt.figure(figsize=(8, 5))
plt.scatter(df['Fuel Consumption Comb (L/100 km)'], df['CO2 Emissions(g/km)'],
            alpha=0.3, color='#C05040', s=20)
plt.title('Fuel Consumption (Combined) vs CO2 Emissions', fontsize=14)
plt.xlabel('Fuel Consumption Comb (L/100 km)')
plt.ylabel('CO2 Emissions (g/km)')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — numerical features only
num_cols = ['Engine Size(L)', 'Cylinders',
            'Fuel Consumption City (L/100 km)',
            'Fuel Consumption Hwy (L/100 km)',
            'Fuel Consumption Comb (L/100 km)',
            'Fuel Consumption Comb (mpg)',
            'CO2 Emissions(g/km)']

plt.figure(figsize=(10, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True)
plt.title('Correlation Heatmap — Numerical Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Average CO2 by number of cylinders
plt.figure(figsize=(9, 4))
avg_co2 = df.groupby('Cylinders')['CO2 Emissions(g/km)'].mean().sort_index()
avg_co2.plot(kind='bar', color='#4A8C6A', edgecolor='black')
plt.title('Average CO2 Emissions by Number of Cylinders', fontsize=13)
plt.xlabel('Cylinders')
plt.ylabel('Average CO2 (g/km)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 6 — Prepare Data for ML

In [ ]:
# Features we will use
feature_cols = [
    'Engine Size(L)', 'Cylinders',
    'Fuel Consumption City (L/100 km)',
    'Fuel Consumption Hwy (L/100 km)',
    'Fuel Consumption Comb (L/100 km)',
    'Vehicle Class_encoded',
    'Fuel Type_encoded',
    'Transmission_encoded'
]

X = df[feature_cols]
y = df['CO2 Emissions(g/km)']

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Training samples:", X_train.shape[0])
print("Testing samples: ", X_test.shape[0])

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("Scaling done!")

## Step 7 — Train ML Models

### Model 1 — Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print("Linear Regression Results:")
print("  MAE :", round(mean_absolute_error(y_test, y_pred_lr), 3))
print("  RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_lr)), 3))
print("  R²  :", round(r2_score(y_test, y_pred_lr), 4))

### Model 2 — Ridge Regression

Same as Linear Regression but with regularisation to avoid overfitting.

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)

print("Ridge Regression Results:")
print("  MAE :", round(mean_absolute_error(y_test, y_pred_ridge), 3))
print("  RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_ridge)), 3))
print("  R²  :", round(r2_score(y_test, y_pred_ridge), 4))

### Model 3 — Decision Tree Regressor

In [ ]:
dt = DecisionTreeRegressor(max_depth=8, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree Regressor Results:")
print("  MAE :", round(mean_absolute_error(y_test, y_pred_dt), 3))
print("  RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_dt)), 3))
print("  R²  :", round(r2_score(y_test, y_pred_dt), 4))

### Model 4 — Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Regressor Results:")
print("  MAE :", round(mean_absolute_error(y_test, y_pred_rf), 3))
print("  RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_rf)), 3))
print("  R²  :", round(r2_score(y_test, y_pred_rf), 4))

## Step 8 — Compare All Models

In [ ]:
results = {
    'Linear Regression': y_pred_lr,
    'Ridge Regression':  y_pred_ridge,
    'Decision Tree':     y_pred_dt,
    'Random Forest':     y_pred_rf,
}

summary = []
for name, pred in results.items():
    summary.append({
        'Model': name,
        'MAE':   round(mean_absolute_error(y_test, pred), 2),
        'RMSE':  round(np.sqrt(mean_squared_error(y_test, pred)), 2),
        'R2':    round(r2_score(y_test, pred), 4),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

In [ ]:
# R² score bar chart
plt.figure(figsize=(9, 5))
colors = ['#4A6C8C', '#C8906A', '#4A8C6A', '#8C6A9E']
bars = plt.bar(summary_df['Model'], summary_df['R2'], color=colors, edgecolor='black')

for bar, val in zip(bars, summary_df['R2']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             str(val), ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.title('Model Comparison — R² Score (higher is better)', fontsize=13)
plt.ylabel('R² Score')
plt.ylim(0, 1.1)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

In [ ]:
# RMSE comparison
plt.figure(figsize=(9, 5))
bars = plt.bar(summary_df['Model'], summary_df['RMSE'], color=colors, edgecolor='black')

for bar, val in zip(bars, summary_df['RMSE']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.title('Model Comparison — RMSE (lower is better)', fontsize=13)
plt.ylabel('RMSE')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## Step 9 — Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, pred), color in zip(axes, 
    [('Linear Regression', y_pred_lr), ('Random Forest', y_pred_rf)],
    ['#4A6C8C', '#4A8C6A']):
    ax.scatter(y_test, pred, alpha=0.3, color=color, s=15)
    # Perfect prediction line
    mn = min(y_test.min(), pred.min())
    mx = max(y_test.max(), pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_title(f'{name}\nActual vs Predicted', fontsize=12)
    ax.set_xlabel('Actual CO2 (g/km)')
    ax.set_ylabel('Predicted CO2 (g/km)')
    ax.legend()

plt.tight_layout()
plt.show()

## Step 10 — Feature Importance (Random Forest)

In [ ]:
importances = rf.feature_importances_
feat_df = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='#C8906A', edgecolor='black')
plt.title('Feature Importance — Random Forest', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## Summary

| Model | Strength |
|---|---|
| Linear Regression | Fast, interpretable, good baseline |
| Ridge Regression | Same as LR but more stable with correlated features |
| Decision Tree | Non-linear, easy to understand, can overfit |
| Random Forest | Best accuracy, handles feature interactions well |

**Key takeaway:** Fuel Consumption (Combined) is by far the strongest predictor of CO2 emissions — which makes physical sense since burning more fuel directly produces more CO2.